# Civil Code Tree Traversal Examples

This notebook shows how to use the LLM-guided traversal flow on the Civil Code sample tree.

It is intentionally example-oriented:
- no gold paths
- no evaluation metrics
- no tests
- just load a tree, run example queries, inspect the traversal, and visualize the prediction tree

Before running the traversal cells, make sure the API key for your selected LLM backend is available in the environment.


In [1]:
from __future__ import annotations

import os
import pickle
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists() and (current / "trees").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from google.genai import types as genai_types

from hyperparams import HyperParams
from llm_apis import GenAIAPI, OpenAIResponsesAPI, VllmAPI
from prompts import get_traversal_prompt_response_constraint
from tree_construction.build_llm_bottom_up_tree import run_coro_sync
from tree_objects import InferSample, SemanticNode
from utils import compute_node_registry, get_all_leaf_nodes_with_path, post_process, setup_logger, visualize_sample

REPO_ROOT


c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.

WindowsPath('C:/Users/jmgil/Desktop/TESE/LATTICE/llm-guided-retrieval')

In [2]:
# Configuration
#
# TREE_PATH defaults to the LLM-built notebook tree if it exists, otherwise the
# sample tree in src/tree_construction.
default_tree_path = REPO_ROOT / "trees" / "PT" / "codigo_civil_notebook" / "tree-bottom-up-llm.pkl"
if not default_tree_path.exists():
    default_tree_path = REPO_ROOT / "src" / "tree_construction" / "codigo_civil_tree_sample.pkl"

# HyperParams.from_args(...) mirrors the way run.py configures retrieval.
# We pass the required args first, then override/add notebook-specific values.
hp = HyperParams.from_args("--subset fiqa --tree_version civil_code_notebook")
hp.TREE_PATH = str(default_tree_path)
hp.DATASET = "PT"
hp.LLM_API_BACKEND = "openai"  # one of: openai, genai, vllm
hp.LLM = "gpt-4.1"
hp.LLM_API_TIMEOUT = 120
hp.LLM_API_MAX_RETRIES = 4
hp.LLM_MAX_CONCURRENT_CALLS = 4
hp.LLM_API_STAGGERING_DELAY = 0.05
hp.VLLM_BASE_URL = "http://localhost:8000/v1"
hp.REASONING_IN_TRAVERSAL_PROMPT = -1
hp.SUBSET = "fiqa"  # chooses the traversal relevance definition
hp.MAX_BEAM_SIZE = 2
hp.SEARCH_WITH_PATH_RELEVANCE = True
hp.NUM_LEAF_CALIB = 0
hp.RELEVANCE_CHAIN_FACTOR = 0.5
hp.MAX_PROMPT_PROTO_SIZE = None
hp.MAX_DOC_DESC_CHAR_LEN = None

LOG_DIR = REPO_ROOT / "results" / "tree_construction"
LOG_DIR.mkdir(parents=True, exist_ok=True)
logger = setup_logger("civil_code_traversal_notebook", str(LOG_DIR / "civil_code_traversal_notebook.log"))

hp


Log file already exists: C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\results\tree_construction\civil_code_traversal_notebook.log, appending to it.


S=fiqa-TV=civil_code_notebook-TPV=5-RInTP=-1-NumLC=10-PlTau=5.0-RCF=0.5-LlmApiB=openai-Llm=gpt-4.1-NumI=20-NumES=1000-MaxBS=2-TP=C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\trees\PT\codigo_civil_notebook\tree-bottom-up-llm.pkl-LlmApiB=openai-Llm=gpt-4.1-VBUrl=http:____localhost:8000__v1-RInTP=-1-S=fiqa-MaxBS=2-NumLC=0-RCF=0.5

In [3]:
tree_path = Path(hp.TREE_PATH)
tree_obj = pickle.loads(tree_path.read_bytes())
semantic_root_node = SemanticNode().load_dict(tree_obj) if isinstance(tree_obj, dict) else tree_obj
node_registry = compute_node_registry(semantic_root_node)
all_leaf_nodes = get_all_leaf_nodes_with_path(semantic_root_node)

print(f"Loaded tree from: {tree_path}")
print(f"Total nodes in semantic tree: {len(node_registry)}")
print(f"Total leaf nodes: {len(all_leaf_nodes)}")
print("Root description preview:")
print(semantic_root_node.desc[:800])


Loaded tree from: C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\trees\PT\codigo_civil_notebook\tree-bottom-up-llm.pkl
Total nodes in semantic tree: 97
Total leaf nodes: 64
Root description preview:
ROOT Node: This subtree covers foundational, transitional, and historical aspects of the Portuguese Civil Code, including its approval, temporal application, and revocation of prior laws. It addresses general legal principles, sources of law, legal incapacities, property and credit rules, special lease and partnership regimes, guardianship and conservatorship frameworks, and the legislative evolution and constitutional review of specific articles. Use this node for questions about the structure and application of the Civil Code, transitional legal provisions, historical regulations, and the impact of legislative changes.. Key topics: transitional provisions; legal incapacities; universal and family partnerships; guardianship and conservatorship; revocation of Civil Code articles; 

In [4]:
def make_llm_api(hp, logger):
    if hp.LLM_API_BACKEND == "genai":
        llm_api = GenAIAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES)
    elif hp.LLM_API_BACKEND == "openai":
        llm_api = OpenAIResponsesAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES)
    elif hp.LLM_API_BACKEND == "vllm":
        llm_api = VllmAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES, base_url=hp.VLLM_BASE_URL)
    else:
        raise ValueError(f"Unknown backend: {hp.LLM_API_BACKEND}")

    llm_api_kwargs = {
        "max_concurrent_calls": hp.LLM_MAX_CONCURRENT_CALLS,
        "response_mime_type": "application/json",
        "response_schema": get_traversal_prompt_response_constraint(bool(hp.REASONING_IN_TRAVERSAL_PROMPT)),
        "staggering_delay": hp.LLM_API_STAGGERING_DELAY,
        "print_summary_report": False,
        "thinking_config": genai_types.ThinkingConfig(thinking_budget=hp.REASONING_IN_TRAVERSAL_PROMPT),
    }

    if hp.LLM_API_BACKEND == "vllm":
        llm_api_kwargs.pop("response_mime_type")
        llm_api_kwargs.pop("thinking_config")
        llm_api_kwargs.pop("response_schema")

    if hp.LLM_API_BACKEND == "openai":
        llm_api_kwargs.pop("response_mime_type")
        llm_api_kwargs.pop("thinking_config")

    return llm_api, llm_api_kwargs


llm_api, llm_api_kwargs = make_llm_api(hp, logger)
print(type(llm_api).__name__)


2026-04-29 17:57:06,482 - civil_code_traversal_notebook - INFO - Initialized client for model: gpt-4.1
2026-04-29 17:57:06,882 - civil_code_traversal_notebook - INFO - Initialized OpenAI Responses client with model: gpt-4.1


OpenAIResponsesAPI


## Example Queries

Pick one of these or write your own legal query in the next cell.


In [5]:
EXAMPLE_QUERIES = [
    "What does the Civil Code say about when a law starts to be in force?",
    "How does the Civil Code treat the ignorance or misinterpretation of the law?",
    "What are the rules on the application of laws in time?",
    "What does the Civil Code say about the rights of foreigners?",
    "How are gaps in the law integrated under the Civil Code?",
]

for idx, query in enumerate(EXAMPLE_QUERIES):
    print(f"[{idx}] {query}")


[0] What does the Civil Code say about when a law starts to be in force?
[1] How does the Civil Code treat the ignorance or misinterpretation of the law?
[2] What are the rules on the application of laws in time?
[3] What does the Civil Code say about the rights of foreigners?
[4] How are gaps in the law integrated under the Civil Code?


In [6]:
query = EXAMPLE_QUERIES[0]
num_iters = 4
query


'What does the Civil Code say about when a law starts to be in force?'

In [7]:
def make_sample(query: str) -> InferSample:
    return InferSample(
        semantic_root_node,
        node_registry,
        hp=hp,
        logger=logger,
        query=query,
        gold_paths=[],
        excluded_ids_set=set(),
    )


def run_one_iteration(sample: InferSample):
    inputs = sample.get_step_prompts()
    prompts = [prompt for prompt, _ in inputs]
    slates = [slate for _, slate in inputs]

    raw_responses = run_coro_sync(llm_api.run_batch(prompts, **llm_api_kwargs))
    response_jsons = [post_process(output, return_json=True) for output in raw_responses]
    sample.update(slates, response_jsons)
    return raw_responses, response_jsons


def print_frontier(sample: InferSample):
    print("Current beam state paths:")
    for state_path in sample.beam_state_paths:
        node = state_path[-1]
        print(f"  path={node.path} | path_relevance={node.path_relevance:.3f} | desc={node.desc[:180]}")


def print_top_predictions(sample: InferSample, k: int = 5):
    top_preds = sample.get_top_predictions(k=k)
    if not top_preds:
        print("No leaf predictions yet. Increase num_iters.")
        return

    for rank, (node, score) in enumerate(top_preds, start=1):
        print(f"[{rank}] path={node.path} | score={score:.3f} | id={node.id}")
        print(node.desc[:600])
        print("-" * 120)


def run_query(query: str, num_iters: int = 4) -> InferSample:
    sample = make_sample(query)
    print(f"Running traversal for query: {query}")
    for step in range(num_iters):
        print(f"\n--- Iteration {step + 1} ---")
        _, response_jsons = run_one_iteration(sample)
        print_frontier(sample)
        if response_jsons:
            first_reasoning = response_jsons[0].get("reasoning", "")
            print("\nReasoning preview:")
            print(str(first_reasoning)[:800])
    return sample


In [8]:
sample = run_query(query, num_iters=num_iters)


2026-04-29 17:57:18,469 - civil_code_traversal_notebook - INFO - Running a batch of 1 prompts...
2026-04-29 17:57:18,473 - civil_code_traversal_notebook - INFO - Concurrency limited to 4 parallel calls.


Running traversal for query: What does the Civil Code say about when a law starts to be in force?

--- Iteration 1 ---


Processing batch: 100%|██████████| 1/1 [00:08<00:00,  8.27s/it, errors=0, active=0, completed=1, 429s=0, 503s=0]
2026-04-29 17:57:30,026 - civil_code_traversal_notebook - INFO - Running a batch of 2 prompts...
2026-04-29 17:57:30,031 - civil_code_traversal_notebook - INFO - Concurrency limited to 4 parallel calls.


Current beam state paths:
  path=(0,) | path_relevance=1.000 | desc=Portuguese Civil Code: General and Transitional Rules. This subtree addresses foundational and transitional aspects of the Portuguese Civil Code, including its approval, temporal a
  path=(3,) | path_relevance=0.650 | desc=Revogação do CC Art. 2.º (Assentos). Informações sobre a revogação do artigo 2.º do Código Civil português relativo aos assentos, incluindo fundamentos legais, decisões de inconsti

Reasoning preview:
The essential problem in the query is to identify what the Portuguese Civil Code specifically states about when a law starts to be in force: that is, the provisions governing the temporal application of laws—the moment a law enters into force. Now, considering each candidate: 

[0] covers the General and Transitional Rules of the Portuguese Civil Code, specifically mentioning 'temporal application', 'transitional provisions', and the approval and revocation of prior laws. This node directly addresses wh

Processing batch: 100%|██████████| 2/2 [00:04<00:00,  2.40s/it, errors=0, active=0, completed=2, 429s=0, 503s=0]
2026-04-29 17:57:35,225 - civil_code_traversal_notebook - INFO - Running a batch of 2 prompts...
2026-04-29 17:57:35,228 - civil_code_traversal_notebook - INFO - Concurrency limited to 4 parallel calls.


Current beam state paths:
  path=(0, 0) | path_relevance=1.000 | desc=Transitional Provisions and Early Application of Portuguese Civil Code. This subtree provides information on the transitional and initial application rules of the Portuguese Civil 
  path=(0, 4) | path_relevance=0.575 | desc=Portuguese Civil Code: General Principles. This subtree provides information on foundational principles and sources of law in the Portuguese Civil Code, including immediate sources

Reasoning preview:
1. The essential problem in the user's query is determining what the Portuguese Civil Code states about the moment when a law comes into effect (its official commencement or entry into force). This requires information about the applicability and effective date of laws as determined by the Civil Code.

2. Candidate Analyses:
- [0] Transitional Provisions and Early Application of Portuguese Civil Code: This candidate specifically mentions rules about the commencement dates of the Code, approval, and 

Processing batch: 100%|██████████| 2/2 [00:06<00:00,  3.14s/it, errors=0, active=0, completed=2, 429s=0, 503s=0]
2026-04-29 17:57:41,790 - civil_code_traversal_notebook - INFO - Running a batch of 2 prompts...
2026-04-29 17:57:41,791 - civil_code_traversal_notebook - INFO - Concurrency limited to 4 parallel calls.


Current beam state paths:
  path=(0, 0, 1) | path_relevance=1.000 | desc=Código Civil: Vigência e Aplicação no Tempo. This subtree addresses questions about the commencement, temporal application, and transitional provisions of the Portuguese Civil Code
  path=(0, 0, 2) | path_relevance=0.975 | desc=Código Civil: Início de Vigência. Este ramo cobre informações sobre o início da vigência do Código Civil português, incluindo datas de entrada em vigor para diferentes artigos e re

Reasoning preview:
1. The essential problem in the query is to find what the Civil Code states about when a law begins to be in force—i.e., the official commencement or entry into effect of a law, according to the Portuguese Civil Code.

2. Candidate reviews:
- [0] Código Civil: Disposições Iniciais. This subtree focuses on the legal basis for the Code's enactment, revocation of prior law, and transitional provisions. Although it touches on general approval and transition, it doesn't specifically focus on the ru

Processing batch: 100%|██████████| 2/2 [00:09<00:00,  4.93s/it, errors=0, active=0, completed=2, 429s=0, 503s=0]


Current beam state paths:
  path=(0, 0, 0) | path_relevance=0.700 | desc=Código Civil: Disposições Iniciais. This subtree covers the initial articles of the Portuguese Civil Code, including its approval, the revocation of previous civil law, and the han
  path=(1,) | path_relevance=0.550 | desc=Universal and Family Partnerships Law. Covers legal frameworks, transitional rules, and historical context for universal and family partnerships under Article 9 of the Civil Code, 

Reasoning preview:
1. The essential problem in the query is to find out what the Civil Code says about the exact start (commencement) of the force of law – i.e., when a law begins to be effective according to the Civil Code.

2. Step-by-step analysis of each passage:

[0]. 'CC Art. 2.º (Começo de vigência)' - This is only a heading (title) and does not include the substantive text of the article. While it signposts where to find the rule, by itself it provides no specific information about when a law starts to be in 

In [9]:
print_top_predictions(sample, k=5)


[1] path=(0, 0, 2, 0) | score=0.962 | id=1b97b71e885d18359d6c7bb9e05c1f311efcfdc0
CC Art. 2.º, n.º 1 (Começo de vigência) O Código Civil entra em vigor no continente e ilhas adjacentes no dia 1 de Junho de 1967, à excepção do disposto nos artigos 1841.º a 1850.º, que começará a vigorar sômente em 1 de Janeiro de 1968.
------------------------------------------------------------------------------------------------------------------------
[2] path=(0, 0, 2, 1) | score=0.603 | id=d5872cfa81c77db2567b55d7055e62e085adf6e6
CC Art. 2.º, n.º 2 (Começo de vigência) O código não é, porém, aplicável às acções que estejam pendentes nos tribunais no dia da sua entrada em vigor, salvo o disposto nos artigos 17.º e 21.º do presente decreto-lei.
------------------------------------------------------------------------------------------------------------------------
[3] path=(0, 0, 1, 2) | score=0.574 | id=1b6a8498d5cfa30f8d34cdcfc67e87db2b47521d
CC Art. 5.º (Aplicação no tempo) A aplicação das disposiç

In [10]:
# Optional: visualize the prediction tree explored by the LLM.
VIS_DIR = REPO_ROOT / "results" / "tree_construction"
VIS_DIR.mkdir(parents=True, exist_ok=True)
html_path = VIS_DIR / "civil_code_traversal_prediction_tree.html"

fig = visualize_sample(sample, width=1400, height=900, save_path=str(html_path))
print(html_path)


Saved plot HTML to C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\results\tree_construction\civil_code_traversal_prediction_tree.html


C:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\results\tree_construction\civil_code_traversal_prediction_tree.html


## Notes

- `num_iters` controls how deep the traversal goes before you inspect results.
- If `print_top_predictions(...)` says there are no leaf predictions yet, increase `num_iters`.
- `hp.SUBSET` controls the relevance definition used inside the traversal prompt. For legal QA-style usage, `fiqa` is usually a reasonable default here.
- `visualize_sample(...)` shows the explored prediction tree, not the static semantic tree structure.
